# 03 – Thí nghiệm can thiệp cho giả thuyết H (32 run, 2 shard)
Thiết kế và quy tắc quyết định đã đăng ký trước trong `pilot/INTERVENTION.md`.

**Cài đặt notebook:** Accelerator = **GPU T4 x2**; **Internet = On**; *Add Input* → output của notebook `01` (`cremad_processed.tar`). Dùng lại notebook 01 đã có, không cần chạy lại.

**Cách chạy (2 notebook song song):** tạo 2 bản của notebook này. Bản thứ nhất đặt `SHARD = 0` (fold 0–1), bản thứ hai đặt `SHARD = 1` (fold 2–3), rồi *Save Version → Save & Run All* cả hai. Mỗi shard có 16 job, khoảng 7–8 giờ.

**Khi xong:** tải `runs_interv_shard{SHARD}.tar` ở tab *Output* của từng bản và gửi cho Claude. Nếu dòng cuối log báo `shard jobs still missing: N` với N > 0: upload file tar đó thành một Dataset, gắn làm input, rồi chạy lại đúng shard đó.

In [ ]:
SHARD = 0             # 0 (fold 0-1) hoặc 1 (fold 2-3)
N_SHARDS = 2          # giữ nguyên
TIME_BUDGET_H = 10.0  # hard stop = +0,5 giờ (giới hạn session Kaggle: 11 giờ)
GRID_ARGS = '--grid intervention'   # giữ nguyên
EXTRA_TRAIN_ARGS = ''               # để trống trên Kaggle

In [ ]:
# Đường dẫn (giữ nguyên trên Kaggle)
WORK = '/tmp'               # đĩa tạm của session: code, data giải nén, run
KIN  = '/kaggle/input'      # input gắn vào notebook (chỉ đọc)
KOUT = '/kaggle/working'    # output được lưu cùng version

In [ ]:
import os, glob, subprocess, sys, json, math, importlib
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
os.chdir(f'{WORK}/icme/pilot')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
print('CPUs:', os.cpu_count())
tars = glob.glob(f'{KIN}/**/cremad_processed.tar', recursive=True)
assert tars, 'Chưa gắn output của notebook 01 làm input'
os.makedirs(f'{WORK}/cremad', exist_ok=True)
sh(f'tar -xf {tars[0]} -C {WORK}/cremad')
os.makedirs(f'{WORK}/runs_interv', exist_ok=True)
for t in glob.glob(f'{KIN}/**/runs_interv_shard*.tar', recursive=True):   # khôi phục các run đã xong (nếu có)
    sh(f'tar -xf {t} -C {WORK}')
    print('restored', t)
WORKERS = max(1, os.cpu_count() // 2)
print('dataloader workers per job:', WORKERS)
sh('nvidia-smi --query-gpu=name,memory.total --format=csv', check=False)

In [ ]:
sh(f'python make_jobs.py --root {WORK}/cremad --out {WORK}/runs_interv --jobs {WORK}/jobs_interv.txt {GRID_ARGS} '
   f'--extra "--amp --workers {WORKERS} {EXTRA_TRAIN_ARGS}"')
sh(f'python launch.py --jobs {WORK}/jobs_interv.txt --gpus 0,1 --per_gpu 1 --shard {SHARD} --n_shards {N_SHARDS} '
   f'--time_budget_h {TIME_BUDGET_H} --hard_stop_h {TIME_BUDGET_H + 0.5} --snapshot_tar {KOUT}/runs_interv_shard{SHARD}.tar')
sh(f"tar -cf {KOUT}/runs_interv_shard{SHARD}.tar -C {WORK} --exclude='*.pt' runs_interv")
sh(f'python analyze_intervention.py --root {WORK}/cremad --runs {WORK}/runs_interv --out {KOUT}/intervention_report_partial.md', check=False)
sh(f'ls -la {KOUT}')